# Radiometry Estimates
Current problem is that my radiometry is code is not agreeing with my Curio graphs. I think I'll see what I can do here to figure out what's reasonable and what isn't.  

## By Hand Writing New Functions
Checking that the Mathemtica SNR agrees with a re-calculation by "first principles"

In [1]:
import math
from math import pi, sqrt
import plotly.express as px
import numpy as np # Used here to generate ordered data easily
from importlib import reload

In [2]:
def mag(x):
    return(-2.5 * math.log10(x))
def amag(x):
    return(10**(-0.4 * x))

In [3]:
cal = 866000 # photons/cm^2/sec
back = 21.61 # Magnitudes per square arcsecond

In [4]:
def signal(x, aper, integrationTime, q):
    ''' x is the source in magnitudes
    aper is the aperture size in m
    IntegrationTime is in seconds, 
    q is the net quantum efficiency'''
    area = math.pi * (aper*100/2)**2
    return(  amag(x) * cal *  area * integrationTime * q  )

In [5]:
def noise(aper, photaper, integrationTime, q):
    '''
    aper is the aperture of the telescope in meters
    photaper is the size of the photometry aperture RADIUS in arcseconds (assumed to be square)
    integration time is in seconds
    calibration and background are picked up from the notebook
    q is the net quantum efficiency 
    '''
    area = math.pi * (aper*100/2)**2
    print('    aperture area', area)
    photarea = math.pi * (photaper/2)**2
    print('    photarea', photarea)
    return( math.sqrt(amag(back) * cal * area *integrationTime * q * photarea ))

In [6]:
print('signal', signal(20.5, 1, 10, 0.2))
print('noise', noise(1, 2, 10, 0.2))
print('snr', signal(20.5, 1, 10, 0.2)  / noise(1, 2, 10, 0.2)  )

signal 85.82973448778654
    aperture area 7853.981633974483
    photarea 3.141592653589793
noise 9.849038176984214
    aperture area 7853.981633974483
    photarea 3.141592653589793
snr 8.714529575929381


This section basically did an recalculation from first principles of the SNR to compare withe the value we seem to have in the Mathematica code.  Turns out to be 8.7 vs 10. for the SNR,  but the value of 20.5 was just eyeballed from the graph, and the calibraiton is from a different source and in different units, so I think good agreement.

## Checking the equation in the Equations Document 

Here I'm calculating the required integration time using  3 from the Equations document.  It should end up being about 10 if I have the numbers right in the Mathematica notebook as used above.  Really this is just a check that equation 3 is derived correcdtly.

In [7]:
pi = math.pi
radperster = 2 * pi / (3600 * 360)
print(radperster)

4.84813681109536e-06


Matching the parameters used in the last section, although converted to m^2.

In [8]:
gamma = 8.71 # This is the SNR we got in the last section
A = math.pi * (100/2)**2
beta = amag(back) * cal / (radperster**2)
omega = pi * (radperster**2)
alpha = amag(20.5) * cal 
eta = 0.2
f = 1
print('\u03B3 gamma', f"{gamma:.2e}")
print('\u03B2 beta',f"{beta:.2e}")
print('\u03c9 omega', f"{omega:.2e}")
print('\u03B1 alpha \u03B1', f"{alpha:.2e}")
print('A', f"{A:.2e}")
print('\u03B7 eta', eta)
print('f', f)
print( 'Integration Time' , ( (gamma**2) * beta * omega) /( (alpha**2) * A * eta * f**2))

γ gamma 8.71e+00
β beta 8.36e+07
ω omega 7.38e-11
α alpha α 5.46e-03
A 7.85e+03
η eta 0.2
f 1
Integration Time 9.989607244511332


This does compare with the last section.

In [9]:
print(alpha * A * 10 * eta)

85.82973448778654


In [10]:
print(sqrt(beta * A * 10 * 0.2 * eta * omega))

4.404623775345462


## Using these values to get S N and SNR

In [11]:
print(signal(20.5, 1, 10, 0.2))
print(noise(1, 1, 10, 0.2))
print(signal(20.5, 1, 10, 0.2)  / noise(1, 2, 10, 0.2)  )

85.82973448778654
    aperture area 7853.981633974483
    photarea 0.7853981633974483
4.924519088492107
    aperture area 7853.981633974483
    photarea 3.141592653589793
8.714529575929381


In [12]:
print(noise(1, 1, 10, 0.2))

    aperture area 7853.981633974483
    photarea 0.7853981633974483
4.924519088492107


In [13]:
print(signal(20.5, 1, 10, 0.2)  / noise(1, 2, 10, 0.2)  )

    aperture area 7853.981633974483
    photarea 3.141592653589793
8.714529575929381


## Discussion
This result is pretty much with the Mathematica code gives, seeing as I'm reading things off the figure by eye. 

**I note that I was using the photomotery RADIUS instead of the DIAMETER in mathematica, somewhat unexpected.**

## Conclusion
My old mathematica calculation and this much rougher calculation agree.  Now what's going on with my python code?

## Testing the main code.  
Most of this is in detector.py which relies on constants.py and rediometry_data.py
Edite detector.py to print out the variables so I can compare directly to the above.

A differences is that we are using square arcseconds instead of steradians.

In [14]:
import detector as de
reload(de)
filt,d = de.makeDetector(1,"V", 1 * de.DEGREE, 2 * de.ARCSEC, 1, qe = 0.2, photfrac = 1)
print(d)
print(de.requiredIntegrationTime(20.5, 10, "V", d, debug=1))

[[7.85398163e-01 7.38413463e-11 2.00000000e-01 1.00000000e+00
  3.08641975e-07 3.49065850e-01 1.74532925e-01 2.61799388e-01
  6.49888933e+11 8.79000000e+09 0.00000000e+00]]
gamma 1.00e+01
beta is, 6.50e+11
omega [7.38413463e-11]
alpha is 5.55e+01
A [0.78539816]
eta [0.2]
f [1.]
[9.93210085]


In [15]:
print(de.testdetector())

gamma 1.00e+01
beta is, 6.50e+11
omega [7.38413463e-11]
alpha is 5.55e+01
A [0.78539816]
eta [0.2]
f [1.]
[9.93210085]
None


This is from the run above doing things by hand:
- γ gamma 8.71e+00
- β beta 8.36e+07
- ω omega 7.38e-11
- α alpha α 5.46e-03
- A 7.85e+03
- η eta 0.2
- f 1
- Integration Time 9.989607244511332


This fits so far, but there some differences that I don't understand:

## Comparing & Conclusion:
- gamma: 10 vs. 8.71 - that's set , OK,
- $\eta$, $f$, $A$ are OK
- $\omega$ is OK
- $\alpha$ is off by $\sim 10^4$ in the python code
- $\beta$ is off by $\sim 10^4$ in the python code actually
- $A$ is too small by a factor of $10^{-4}$ in the code

Since $\alpha$ is used twice in the denomintor, $A$ once in the denominator, and $\beta$ once in the numerator all these things cancel and we get the right answer.  The difference is in fact explained by the fact that we are using square meters in the code, and square cm in the test code we have here. 

# Thinking about time steps
If the integration time might be large enough, perhaps we can just do small timesteps and move the FOV at the end of each step.  So how long is required integration time for say a 1m looing to 20th magnitude?

In [16]:
?de.makeDetector

Signature:
de.makeDetector(
    n,
    band,
    fov,
    ifov,
    aper,
    qe=0.5,
    photfrac=0.7,
    solarex=0.3490658503988659,
    lunarex=0.17453292519943295,
    earthex=0.2617993877991494,
)
Docstring:
makeDetector takes parameters of a sensor and stuffs a filter array and a detector array, which it returns.
n is the number of sensors to produce
band is the band the measurement takes place in (see radiometry_data)
fov is the field of view- assumed square- in radians
ifov is the pixel fov - assumed square - in radians
aper is the aperture diameter - assumed round - in meters
qe is th quantum efficiency of the system from entrance aperture
    to detectro
photfrac is the fraction of the light captured in the photometry aperture
solarex is the solar exclusion angle in radians
lunarex is the lunar exclusion angle in radians
earthex is the earth exclusion angle in radians

This function is called when a new satellite is created.
It uses the data from FILTER_DATA in radiometry_da

In [17]:
?de.requiredIntegrationTime

Signature: de.requiredIntegrationTime(limitingMag, SNR, filt, d, debug=0)
Docstring:
requiredIntegrationTime(limitingMag, d)
takes a two dimensional detector array ("detect")and calculates
      all the integration tiemes
and returns those as a vector.
For comparison with the equations paper, we first extract the variables
to the conventional names used in that paper.
File:      ~/Desktop/vibevolts/detector.py
Type:      function

In [39]:
import detector as de
reload(de)
filt,d = de.makeDetector(1,"V", 3 * de.DEGREE, 5 * de.ARCSEC, 0.4, qe = 0.5, photfrac = 0.5)
print(de.requiredIntegrationTime(20.5, 10, "V", d))

[620.75630296]


In [4]:
600 * 15 / 3600

2.5